# Public research notebook

This notebook is a cleaned public version of the original
research workflow.

## Execution model

- All filesystem paths are relative to the repository root.
- No external mounted filesystem is required.
- Stored cell outputs have been removed.
- Generated files are written below the local `results/`
  directory.
- The archival source notebook remains unchanged.


In [ ]:
# Portable repository configuration
#
# The notebook assumes that it is executed from the repository
# root or from a cloned copy of the repository.

from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()

# Move upward when the notebook is launched from a nested folder.
if REPOSITORY_ROOT.name in {
    "lorenz",
    "rossler",
    "duffing",
    "kuramoto",
    "stuart_landau",
    "coupled_map_lattice",
}:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parents[1]

DATA_DIR = REPOSITORY_ROOT / "data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIR}")


# Lorenz System — Dynamic Compatibility Window

This notebook reproduces the Lorenz-system analyses presented in *Dynamic Compatibility and Reorganization Across Nonlinear Systems*.

It evaluates the effect of the compatibility threshold $\Delta$ on:

- transitions between the two wings of the Lorenz attractor,
- dwell time,
- phase-space exploration,
- run-to-run variability,
- correlations between observables,
- damping sensitivity,
- sensitivity to the parameter $\rho$,
- normalized compatibility thresholds.

**Publication fidelity:** the computational cells below are copied from the original research notebook without changes to the numerical logic.


In [ ]:
# ============================================================
# DELTA WINDOW PROJECT — UNIVERSAL PIPELINE INIT
# ============================================================

# -----------------------------
# GOOGLE DRIVE
# -----------------------------

# -----------------------------
# IMPORTS
# -----------------------------
import os
import shutil
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime

# -----------------------------
# TIMESTAMP
# -----------------------------
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# -----------------------------
# PROJECT NAME
# CHANGE FOR EACH NOTEBOOK
# -----------------------------
PROJECT_NAME = "delta_window_lorenz_v2"

# Examples:
# delta_window_rossler_v2
# delta_window_duffing_v2

# -----------------------------
# BASE DIRECTORY
# -----------------------------
BASE_DIR = "data/raw"

PROJECT_DIR = f"{BASE_DIR}/{PROJECT_NAME}"

CSV_DIR = f"{PROJECT_DIR}/csv"
FIG_DIR = f"{PROJECT_DIR}/figures"
FINAL_FIG_DIR = f"{PROJECT_DIR}/final_figures"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
LOG_DIR = f"{PROJECT_DIR}/logs"

# -----------------------------
# CREATE DIRECTORIES
# -----------------------------
ALL_DIRS = [
    PROJECT_DIR,
    CSV_DIR,
    FIG_DIR,
    FINAL_FIG_DIR,
    CHECKPOINT_DIR,
    LOG_DIR
]

for d in ALL_DIRS:
    os.makedirs(d, exist_ok=True)

print("Directories created.")

# -----------------------------
# SAVE FUNCTIONS
# -----------------------------

def save_csv(df, name, folder=CSV_DIR):

    path = f"{folder}/{name}_{TIMESTAMP}.csv"

    df.to_csv(path, index=False)

    print(f"[CSV SAVED]")
    print(path)

    return path


def save_checkpoint(df, name):

    path = f"{CHECKPOINT_DIR}/{name}_CHECKPOINT.csv"

    df.to_csv(path, index=False)

    print(f"[CHECKPOINT SAVED]")
    print(path)

    return path


def save_figure(plt_obj, name, folder=FIG_DIR):

    path = f"{folder}/{name}_{TIMESTAMP}.png"

    plt_obj.savefig(
        path,
        dpi=300,
        bbox_inches='tight'
    )

    print(f"[FIGURE SAVED]")
    print(path)

    return path


def save_final_figure(plt_obj, name):

    path = f"{FINAL_FIG_DIR}/{name}_{TIMESTAMP}.png"

    plt_obj.savefig(
        path,
        dpi=300,
        bbox_inches='tight'
    )

    print(f"[FINAL FIGURE SAVED]")
    print(path)

    return path


# -----------------------------
# SESSION INFO
# -----------------------------
session_info = {
    "project_name": PROJECT_NAME,
    "timestamp": TIMESTAMP
}

with open(f"{LOG_DIR}/session_info_{TIMESTAMP}.json", "w") as f:
    json.dump(session_info, f, indent=4)

print("\n===================================================")
print("DELTA WINDOW PIPELINE INITIALIZED")
print("===================================================")
print(PROJECT_DIR)
print("===================================================")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# USTAWIENIA
# =========================
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0

DT = 0.01
STEPS = 10000
RUNS = 50
DAMPING = 0.5

delta_values = np.arange(0.1, 3.1, 0.1)

# =========================
# SYMULACJA LORENZA + DELTA
# =========================
def simulate(delta, damping=0.5, rng=None,
             sigma=10.0, rho=28.0, beta=8.0/3.0,
             dt=0.01, steps=10000):
    if rng is None:
        rng = np.random.default_rng()

    # lekko losowy start
    x = 1.0 + 0.01 * rng.normal()
    y = 1.0 + 0.01 * rng.normal()
    z = 1.0 + 0.01 * rng.normal()

    xs = [x]
    ys = [y]
    zs = [z]

    filtered_steps = 0

    for _ in range(steps):
        dx = sigma * (y - x)
        dy = x * (rho - z) - y
        dz = x * y - beta * z

        new_x = x + dx * dt
        new_y = y + dy * dt
        new_z = z + dz * dt

        jump = np.sqrt((new_x - x) ** 2 + (new_y - y) ** 2 + (new_z - z) ** 2)

        if jump > delta:
            new_x = x + (new_x - x) * damping
            new_y = y + (new_y - y) * damping
            new_z = z + (new_z - z) * damping
            filtered_steps += 1

        x, y, z = new_x, new_y, new_z

        xs.append(x)
        ys.append(y)
        zs.append(z)

    xs = np.array(xs, dtype=float)
    ys = np.array(ys, dtype=float)
    zs = np.array(zs, dtype=float)

    # Wing transitions correspond to changes in the sign of x
    signs = np.sign(xs)
    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]
    if signs[0] == 0:
        signs[0] = 1

    transitions = np.sum(signs[1:] != signs[:-1])

    return transitions, filtered_steps, xs, ys, zs

# =========================
# DWELL TIME
# =========================
def get_dwell_times_from_x(xs):
    xs = np.array(xs, dtype=float)
    signs = np.sign(xs)

    if len(signs) == 0:
        return np.array([])

    for i in range(1, len(signs)):
        if signs[i] == 0:
            signs[i] = signs[i - 1]
    if signs[0] == 0:
        signs[0] = 1

    dwell_times = []
    current_len = 1

    for i in range(1, len(signs)):
        if signs[i] == signs[i - 1]:
            current_len += 1
        else:
            dwell_times.append(current_len)
            current_len = 1

    dwell_times.append(current_len)
    return np.array(dwell_times)

# =========================
# TEST: DWELL TIME vs DELTA
# =========================
rows = []

for delta in delta_values:
    transitions_runs = []
    filtered_runs = []
    mean_dwell_runs = []
    median_dwell_runs = []
    n_dwells_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        transitions, filtered_steps, xs, ys, zs = simulate(
            delta=delta,
            damping=DAMPING,
            rng=rng,
            sigma=sigma,
            rho=rho,
            beta=beta,
            dt=DT,
            steps=STEPS
        )

        dwell_steps = get_dwell_times_from_x(xs)

        transitions_runs.append(transitions)
        filtered_runs.append(filtered_steps)

        if len(dwell_steps) > 0:
            mean_dwell_runs.append(np.mean(dwell_steps) * DT)
            median_dwell_runs.append(np.median(dwell_steps) * DT)
            n_dwells_runs.append(len(dwell_steps))
        else:
            mean_dwell_runs.append(np.nan)
            median_dwell_runs.append(np.nan)
            n_dwells_runs.append(0)

    rows.append({
        "delta": delta,
        "mean_transitions": np.mean(transitions_runs),
        "std_transitions": np.std(transitions_runs),
        "mean_filtered_steps": np.mean(filtered_runs),
        "std_filtered_steps": np.std(filtered_runs),
        "mean_dwell_time": np.nanmean(mean_dwell_runs),
        "std_dwell_time": np.nanstd(mean_dwell_runs),
        "median_dwell_time": np.nanmean(median_dwell_runs),
        "mean_number_of_dwells": np.mean(n_dwells_runs),
    })

df_results = pd.DataFrame(rows)

print("Results table:")
display(df_results)

# =========================
# plot 1: transitions vs delta
# =========================
plt.figure(figsize=(8, 5))
plt.errorbar(
    df_results["delta"],
    df_results["mean_transitions"],
    yerr=df_results["std_transitions"],
    fmt='o-',
    capsize=4
)
plt.xlabel("Delta")
plt.ylabel("Mean transitions")
plt.title("Mean transitions vs Delta")
plt.grid(True)
plt.show()

# =========================
# plot 2: dwell time vs delta
# =========================
plt.figure(figsize=(8, 5))
plt.errorbar(
    df_results["delta"],
    df_results["mean_dwell_time"],
    yerr=df_results["std_dwell_time"],
    fmt='o-',
    capsize=4
)
plt.xlabel("Delta")
plt.ylabel("Mean dwell time")
plt.title("Mean dwell time vs Delta")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# TEST 3: PHASE-SPACE EXPLORATION vs DELTA
# =========================

def phase_space_occupancy(xs, zs, bins_x=60, bins_z=60):
    xs = np.asarray(xs, dtype=float)
    zs = np.asarray(zs, dtype=float)

    H, xedges, zedges = np.histogram2d(xs, zs, bins=[bins_x, bins_z])
    occupied = np.sum(H > 0)
    return occupied

rows_exploration = []

for delta in delta_values:
    occupancy_runs = []
    range_z_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        transitions, filtered_steps, xs, ys, zs = simulate(
            delta=delta,
            damping=DAMPING,
            rng=rng,
            sigma=sigma,
            rho=rho,
            beta=beta,
            dt=DT,
            steps=STEPS
        )

        occ = phase_space_occupancy(xs, zs, bins_x=60, bins_z=60)
        rz = np.max(zs) - np.min(zs)

        occupancy_runs.append(occ)
        range_z_runs.append(rz)

    rows_exploration.append({
        "delta": delta,
        "mean_occupancy": np.mean(occupancy_runs),
        "std_occupancy": np.std(occupancy_runs),
        "mean_range_z": np.mean(range_z_runs),
        "std_range_z": np.std(range_z_runs),
    })

df_exploration = pd.DataFrame(rows_exploration)

print("Tabela exploration:")
display(df_exploration)

# plot 1: occupancy vs delta
plt.figure(figsize=(8, 5))
plt.errorbar(
    df_exploration["delta"],
    df_exploration["mean_occupancy"],
    yerr=df_exploration["std_occupancy"],
    fmt='o-',
    capsize=4
)
plt.xlabel("Delta")
plt.ylabel("Mean occupied bins in (x,z)")
plt.title("Phase-space exploration vs Delta")
plt.grid(True)
plt.show()

# plot 2: range_z vs delta
plt.figure(figsize=(8, 5))
plt.errorbar(
    df_exploration["delta"],
    df_exploration["mean_range_z"],
    yerr=df_exploration["std_range_z"],
    fmt='o-',
    capsize=4
)
plt.xlabel("Delta")
plt.ylabel("Mean range(z)")
plt.title("Vertical extent vs Delta")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# TEST 4: RUN-TO-RUN VARIANCE vs DELTA
# =========================

rows_variance = []

for delta in delta_values:
    transitions_runs = []
    dwell_runs = []
    occupancy_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        transitions, filtered_steps, xs, ys, zs = simulate(
            delta=delta,
            damping=DAMPING,
            rng=rng,
            sigma=sigma,
            rho=rho,
            beta=beta,
            dt=DT,
            steps=STEPS
        )

        # Dwell time for this run
        dwell_steps = get_dwell_times_from_x(xs)
        if len(dwell_steps) > 0:
            mean_dwell = np.mean(dwell_steps) * DT
        else:
            mean_dwell = np.nan

        # Exploration metric for this run
        occ = phase_space_occupancy(xs, zs, bins_x=60, bins_z=60)

        transitions_runs.append(transitions)
        dwell_runs.append(mean_dwell)
        occupancy_runs.append(occ)

    transitions_runs = np.array(transitions_runs, dtype=float)
    dwell_runs = np.array(dwell_runs, dtype=float)
    occupancy_runs = np.array(occupancy_runs, dtype=float)

    rows_variance.append({
        "delta": delta,

        "var_transitions": np.nanvar(transitions_runs),
        "std_transitions": np.nanstd(transitions_runs),
        "cv_transitions": np.nanstd(transitions_runs) / np.nanmean(transitions_runs),

        "var_dwell": np.nanvar(dwell_runs),
        "std_dwell": np.nanstd(dwell_runs),
        "cv_dwell": np.nanstd(dwell_runs) / np.nanmean(dwell_runs),

        "var_occupancy": np.nanvar(occupancy_runs),
        "std_occupancy": np.nanstd(occupancy_runs),
        "cv_occupancy": np.nanstd(occupancy_runs) / np.nanmean(occupancy_runs),
    })

df_variance = pd.DataFrame(rows_variance)

print("Tabela variance:")
display(df_variance)

# =========================
# plot 1: STD transitions vs Delta
# =========================
plt.figure(figsize=(8, 5))
plt.plot(df_variance["delta"], df_variance["std_transitions"], 'o-')
plt.xlabel("Delta")
plt.ylabel("STD of transitions")
plt.title("Run-to-run variability of transitions vs Delta")
plt.grid(True)
plt.show()

# =========================
# plot 2: STD dwell time vs Delta
# =========================
plt.figure(figsize=(8, 5))
plt.plot(df_variance["delta"], df_variance["std_dwell"], 'o-')
plt.xlabel("Delta")
plt.ylabel("STD of mean dwell time")
plt.title("Run-to-run variability of dwell time vs Delta")
plt.grid(True)
plt.show()

# =========================
# plot 3: STD occupancy vs Delta
# =========================
plt.figure(figsize=(8, 5))
plt.plot(df_variance["delta"], df_variance["std_occupancy"], 'o-')
plt.xlabel("Delta")
plt.ylabel("STD of occupancy")
plt.title("Run-to-run variability of phase-space exploration vs Delta")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# TEST 5: CORRELATIONS vs DELTA
# =========================

rows_corr = []

for delta in delta_values:
    transitions_runs = []
    dwell_runs = []
    occupancy_runs = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)

        transitions, filtered_steps, xs, ys, zs = simulate(
            delta=delta,
            damping=DAMPING,
            rng=rng,
            sigma=sigma,
            rho=rho,
            beta=beta,
            dt=DT,
            steps=STEPS
        )

        # dwell
        dwell_steps = get_dwell_times_from_x(xs)
        if len(dwell_steps) > 0:
            mean_dwell = np.mean(dwell_steps) * DT
        else:
            mean_dwell = np.nan

        # exploration
        occ = phase_space_occupancy(xs, zs, bins_x=60, bins_z=60)

        transitions_runs.append(transitions)
        dwell_runs.append(mean_dwell)
        occupancy_runs.append(occ)

    # konwersja
    t = np.array(transitions_runs, dtype=float)
    d = np.array(dwell_runs, dtype=float)
    o = np.array(occupancy_runs, dtype=float)

    # maska NaN
    mask = (~np.isnan(t)) & (~np.isnan(d)) & (~np.isnan(o))

    if np.sum(mask) > 2:
        corr_td = np.corrcoef(t[mask], d[mask])[0, 1]
        corr_to = np.corrcoef(t[mask], o[mask])[0, 1]
        corr_do = np.corrcoef(d[mask], o[mask])[0, 1]
    else:
        corr_td = np.nan
        corr_to = np.nan
        corr_do = np.nan

    rows_corr.append({
        "delta": delta,
        "corr_transitions_dwell": corr_td,
        "corr_transitions_occupancy": corr_to,
        "corr_dwell_occupancy": corr_do
    })

df_corr = pd.DataFrame(rows_corr)

print("Tabela korelacji:")
display(df_corr)

# =========================
# plots
# =========================

plt.figure(figsize=(8,5))
plt.plot(df_corr["delta"], df_corr["corr_transitions_dwell"], 'o-', label='T vs D')
plt.plot(df_corr["delta"], df_corr["corr_transitions_occupancy"], 'o-', label='T vs O')
plt.plot(df_corr["delta"], df_corr["corr_dwell_occupancy"], 'o-', label='D vs O')
plt.axhline(0, linestyle='--')
plt.xlabel("Delta")
plt.ylabel("Correlation")
plt.title("Correlations vs Delta")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# TEST 6: DAMPING EFFECT
# =========================

damping_values = [0.3, 0.5, 0.8]

results_damping = {}

for damping in damping_values:
    mean_transitions = []
    mean_dwell = []

    for delta in delta_values:
        transitions_runs = []
        dwell_runs = []

        for run in range(RUNS):
            rng = np.random.default_rng(run)

            transitions, filtered_steps, xs, ys, zs = simulate(
                delta=delta,
                damping=damping,
                rng=rng,
                sigma=sigma,
                rho=rho,
                beta=beta,
                dt=DT,
                steps=STEPS
            )

            dwell_steps = get_dwell_times_from_x(xs)

            if len(dwell_steps) > 0:
                mean_d = np.mean(dwell_steps) * DT
            else:
                mean_d = np.nan

            transitions_runs.append(transitions)
            dwell_runs.append(mean_d)

        mean_transitions.append(np.nanmean(transitions_runs))
        mean_dwell.append(np.nanmean(dwell_runs))

    results_damping[damping] = {
        "transitions": np.array(mean_transitions),
        "dwell": np.array(mean_dwell)
    }

# =========================
# plot 1: transitions
# =========================
plt.figure(figsize=(8,5))

for damping in damping_values:
    plt.plot(
        delta_values,
        results_damping[damping]["transitions"],
        label=f"damping={damping}"
    )

plt.xlabel("Delta")
plt.ylabel("Mean transitions")
plt.title("Transitions vs Delta (different damping)")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# plot 2: dwell time
# =========================
plt.figure(figsize=(8,5))

for damping in damping_values:
    plt.plot(
        delta_values,
        results_damping[damping]["dwell"],
        label=f"damping={damping}"
    )

plt.xlabel("Delta")
plt.ylabel("Mean dwell time")
plt.title("Dwell time vs Delta (different damping)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# TEST 7A: DIFFERENT RHO
# =========================

rho_values = [20.0, 28.0, 35.0]

results_rho = {}

for rho_test in rho_values:
    mean_transitions = []
    mean_dwell = []

    for delta in delta_values:
        transitions_runs = []
        dwell_runs = []

        for run in range(RUNS):
            rng = np.random.default_rng(run)

            transitions, filtered_steps, xs, ys, zs = simulate(
                delta=delta,
                damping=DAMPING,
                rng=rng,
                sigma=sigma,
                rho=rho_test,
                beta=beta,
                dt=DT,
                steps=STEPS
            )

            dwell_steps = get_dwell_times_from_x(xs)

            if len(dwell_steps) > 0:
                mean_d = np.mean(dwell_steps) * DT
            else:
                mean_d = np.nan

            transitions_runs.append(transitions)
            dwell_runs.append(mean_d)

        mean_transitions.append(np.nanmean(transitions_runs))
        mean_dwell.append(np.nanmean(dwell_runs))

    results_rho[rho_test] = {
        "transitions": np.array(mean_transitions),
        "dwell": np.array(mean_dwell)
    }

# =========================
# plot 1: transitions
# =========================
plt.figure(figsize=(8,5))

for rho_test in rho_values:
    plt.plot(
        delta_values,
        results_rho[rho_test]["transitions"],
        label=f"rho={rho_test}"
    )

plt.xlabel("Delta")
plt.ylabel("Mean transitions")
plt.title("Transitions vs Delta (different rho)")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# plot 2: dwell time
# =========================
plt.figure(figsize=(8,5))

for rho_test in rho_values:
    plt.plot(
        delta_values,
        results_rho[rho_test]["dwell"],
        label=f"rho={rho_test}"
    )

plt.xlabel("Delta")
plt.ylabel("Mean dwell time")
plt.title("Dwell time vs Delta (different rho)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# TEST 7B: NORMALIZED DELTA
# =========================

rho_values = [20.0, 28.0, 35.0]

def simulate_raw_steps(rng=None, sigma=10.0, rho=28.0, beta=8.0/3.0, dt=0.01, steps=10000):
    """
    Symulacja Lorenza BEZ filtra delta.
    Zwraca długości kolejnych kroków w 3D.
    """
    if rng is None:
        rng = np.random.default_rng()

    x = 1.0 + 0.01 * rng.normal()
    y = 1.0 + 0.01 * rng.normal()
    z = 1.0 + 0.01 * rng.normal()

    jumps = []

    for _ in range(steps):
        dx = sigma * (y - x)
        dy = x * (rho - z) - y
        dz = x * y - beta * z

        new_x = x + dx * dt
        new_y = y + dy * dt
        new_z = z + dz * dt

        jump = np.sqrt((new_x - x) ** 2 + (new_y - y) ** 2 + (new_z - z) ** 2)
        jumps.append(jump)

        x, y, z = new_x, new_y, new_z

    return np.array(jumps, dtype=float)

# 1. Compute the baseline step scale for each rho
baseline_step_scale = {}

for rho_test in rho_values:
    all_means = []

    for run in range(RUNS):
        rng = np.random.default_rng(run)
        jumps = simulate_raw_steps(
            rng=rng,
            sigma=sigma,
            rho=rho_test,
            beta=beta,
            dt=DT,
            steps=STEPS
        )
        all_means.append(np.mean(jumps))

    baseline_step_scale[rho_test] = np.mean(all_means)

print("Baseline mean step scales:")
for rho_test in rho_values:
    print(f"rho={rho_test}: mean_step={baseline_step_scale[rho_test]:.6f}")

# 2. policz transitions i dwell vs delta jak above
results_rho_norm = {}

for rho_test in rho_values:
    mean_transitions = []
    mean_dwell = []

    for delta in delta_values:
        transitions_runs = []
        dwell_runs = []

        for run in range(RUNS):
            rng = np.random.default_rng(run)

            transitions, filtered_steps, xs, ys, zs = simulate(
                delta=delta,
                damping=DAMPING,
                rng=rng,
                sigma=sigma,
                rho=rho_test,
                beta=beta,
                dt=DT,
                steps=STEPS
            )

            dwell_steps = get_dwell_times_from_x(xs)
            if len(dwell_steps) > 0:
                mean_d = np.mean(dwell_steps) * DT
            else:
                mean_d = np.nan

            transitions_runs.append(transitions)
            dwell_runs.append(mean_d)

        mean_transitions.append(np.nanmean(transitions_runs))
        mean_dwell.append(np.nanmean(dwell_runs))

    delta_norm = delta_values / baseline_step_scale[rho_test]

    results_rho_norm[rho_test] = {
        "delta_norm": np.array(delta_norm, dtype=float),
        "transitions": np.array(mean_transitions, dtype=float),
        "dwell": np.array(mean_dwell, dtype=float)
    }

# 3. plot transitions vs znormalizowane delta
plt.figure(figsize=(8,5))

for rho_test in rho_values:
    plt.plot(
        results_rho_norm[rho_test]["delta_norm"],
        results_rho_norm[rho_test]["transitions"],
        label=f"rho={rho_test}"
    )

plt.xlabel("Delta / mean_step")
plt.ylabel("Mean transitions")
plt.title("Transitions vs normalized Delta")
plt.legend()
plt.grid(True)
plt.show()

# 4. plot dwell vs znormalizowane delta
plt.figure(figsize=(8,5))

for rho_test in rho_values:
    plt.plot(
        results_rho_norm[rho_test]["delta_norm"],
        results_rho_norm[rho_test]["dwell"],
        label=f"rho={rho_test}"
    )

plt.xlabel("Delta / mean_step")
plt.ylabel("Mean dwell time")
plt.title("Dwell time vs normalized Delta")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
try:

    np.savez(
        "lorenz_data.npz",
        delta_norm=np.array(delta_norm, dtype=float),
        exploration=np.array(mean_occupancy, dtype=float),
        filtered=np.array(mean_filtered, dtype=float)
    )

    print("lorenz_data.npz saved")

except Exception as e:

    print("Skipping lorenz_data.npz save")
    print(e)

print("saved lorenz_data.npz")

In [ ]:
# ============================================================
# FINAL UNIVERSAL SAVE BLOCK
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import shutil

FINAL_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("\n===================================================")
print("STARTING FINAL AUTO SAVE")
print("===================================================")

# ============================================================
# AUTO SAVE ALL DATAFRAMES
# ============================================================

saved = []

for var_name, var_value in list(globals().items()):

    if isinstance(var_value, pd.DataFrame):

        try:

            save_path = f"{CSV_DIR}/{var_name}_{FINAL_TIMESTAMP}.csv"

            var_value.to_csv(save_path, index=False)

            saved.append(var_name)

            print(f"[DATAFRAME SAVED]")
            print(save_path)

        except Exception as e:

            print(f"[ERROR SAVING {var_name}]")
            print(e)

# ============================================================
# SAVE DATAFRAME INDEX
# ============================================================

index_df = pd.DataFrame({
    "saved_dataframe": saved
})

index_path = f"{CSV_DIR}/RECOVERED_DATAFRAME_INDEX_{FINAL_TIMESTAMP}.csv"

index_df.to_csv(index_path, index=False)

print("\n[INDEX SAVED]")
print(index_path)

# ============================================================
# SAVE ALL OPEN FIGURES
# ============================================================

fig_nums = plt.get_fignums()

print(f"\nOpen figures found: {len(fig_nums)}")

for i, fig_num in enumerate(fig_nums):

    try:

        fig = plt.figure(fig_num)

        fig_path = f"{FIG_DIR}/figure_{i+1}_{FINAL_TIMESTAMP}.png"

        fig.savefig(
            fig_path,
            dpi=300,
            bbox_inches='tight'
        )

        print(f"[FIGURE SAVED]")
        print(fig_path)

    except Exception as e:

        print(f"[ERROR SAVING FIGURE {i+1}]")
        print(e)

# ============================================================
# CREATE FULL ZIP BACKUP
# ============================================================

try:

    zip_path = f"{BASE_DIR}/{PROJECT_NAME}_FULL_BACKUP_{FINAL_TIMESTAMP}"

    shutil.make_archive(
        zip_path,
        'zip',
        PROJECT_DIR
    )

    print("\n[FULL ZIP BACKUP CREATED]")
    print(f"{zip_path}.zip")

except Exception as e:

    print("[ZIP BACKUP ERROR]")
    print(e)

print("\n===================================================")
print("FINAL SAVE COMPLETED")
print("===================================================")